## Automating 2 Hyperparameters

In [1]:
import numpy as np
import pandas as pd
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

In [2]:
data = pd.read_excel('../TreeBasedModels/data/default of credit card clients.xls',
                     header=1,
                     index_col=0)
data.head()

,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_0,PAY_2,PAY_3,PAY_4,PAY_5,PAY_6,BILL_AMT1,BILL_AMT2,BILL_AMT3,BILL_AMT4,BILL_AMT5,BILL_AMT6,PAY_AMT1,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6,default payment next month
ID,,,,,,,,,,,,,,,,,,,,,,,,
1,20000,2,2,1,24,2,2,-1,-1,-2,-2,3913,3102,689,0,0,0,0,689,0,0,0,0,1
2,120000,2,2,2,26,-1,2,0,0,0,2,2682,1725,2682,3272,3455,3261,0,1000,1000,1000,0,2000,1
3,90000,2,2,2,34,0,0,0,0,0,0,29239,14027,13559,14331,14948,15549,1518,1500,1000,1000,1000,5000,0
4,50000,2,2,1,37,0,0,0,0,0,0,46990,48233,49291,28314,28959,29547,2000,2019,1200,1100,1069,1000,0
5,50000,1,2,1,57,-1,0,-1,0,0,0,8617,5670,35835,20940,19146,19131,2000,36681,10000,9000,689,679,0


In [3]:
# Attributes are from column 1 to 24
X = data.iloc[:, 1:24]
y = data['default payment next month']

# Split data in Train and Test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

What about testing values of 2 hyperparameters?

Using a GBM algorithm:

In [4]:
learn_rate_list = [0.001, 0.01, 0.05]
max_depth_list = [4, 6, 8, 10]
# We could use a (nested) for loop!

Firstly a model creation function:

In [14]:
def gbm_grid_search(learn_rate, max_depth):
    model = GradientBoostingClassifier(
        learning_rate=learn_rate,
        max_depth=max_depth)
    
    predictions = model.fit(X_train, y_train).predict(X_test)
    return([learn_rate, max_depth, accuracy_score(y_test, predictions)])

Now we can loop through our list of hyperparameters and call our function:

In [15]:
results_list = []

for learn_rate in learn_rate_list:
    for max_depth in max_depth_list:
        results_list.append(gbm_grid_search(learn_rate, max_depth))

We can put these results into a DataFrame as well and print out:

In [16]:
results_df = pd.DataFrame( results_list, columns=["learning_rate", "mex_depth", "accuracy"])
results_df

,learning_rate,mex_depth,accuracy
0,0.001,4,0.781167
1,0.001,6,0.781167
2,0.001,8,0.781167
3,0.001,10,0.781167
4,0.010,4,1.000000
5,0.010,6,1.000000
6,0.010,8,1.000000
7,0.010,10,1.000000
8,0.050,4,1.000000
9,0.050,6,1.000000


### Grid Search with Scikit-Learn

__Grid Search Object__

Steps in a Grid Search:

* 1.- An algorithm to tune the hyperparameters. (Sometimes called an '_estimator_')
* 2.- Defining which hyperparameters we will tune.
* 3.- Defining a range of values for each hyperparameter.
* 4.- Setting a cross-validation scheme; and
* 5.- Define a score function so we can decide which square on our grid was 'the best'.
* 6.- include extra useful information or functions.

The important inputs are:

__`estimator`__

The `estimator` input:
    - Essencially our algorithm
    - You have already worked with KNN, Random Forest, GBM, Logistic Regression.

__Remember: Only one estimator per GridSearchCV object.__

__`param_grid`__

The `param_grid` input:
    - Setting wich hyperparameters and values to test

Rather than a list:

```Python
max_depth_list = [2, 4, 6, 8]
min_samples_leaf_list = [1, 2, 4, 6]
```

This would be:

```Python
param_grid = {
    'max_depth' : [2, 4, 6, 8],
    'min_samples_leaf': [1, 2, 4, 6]
}
```

__Remember: The keys in your input `param_grid` dictionary must be valid hyperparameters.__

__`cv`__

The `cv` input:

- Choice of how to undertke cross-validation.
- using an integer undertake k-fold cross validation where 5 or 10 is usually standard.


__`scoring`__

The `scoring` input:
- Which score to use to choose the best grid square (model)
- Use your own or Scikit Learn's `metric` module

You can check all the built in scoring functions this way:

```Python
from sklearn import metrics
sorted(metrics.SCORERS.keys())
```

__`refit`__

The `refit` input:
- Fits the best hyperparameters to the training data
- Allows the `GridSearchCV` object to be used as an estimator (for prediction)
- A very handly option!

__`n_jobs`__

The `n_jobs` input:
- Assists with parallel execution
- Allows multiple models to be created at the same time, rather than one after the other

Some handy code:

```Python
import os
print(os.cpu_count())
```

Careful using all your cores for modeling if you want to do ther work!

__`return_train_score`__

The `grid_train_score` input:
- Logs statistics about the training runs that were undertaken
- Useful for analyzing bias-variance trade-off but adds computational expense.
- Does not assist in picking the best mode, only for analysis purposes



__Building a GridSearchCV object__

Building our own GridSearchCV Object:

In [15]:
import os
print(os.cpu_count())

8


In [16]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV

# Create the grid
param_grid = {
    'max_depth': [2, 4, 8],
    'min_samples_leaf': [1, 2, 4],
    'max_features': ['auto', 'sqrt']
}

# Get a base classifier with some set parameters
rf_class = RandomForestClassifier(criterion='entropy')

# Putting the pieces together
grid_rf_class = GridSearchCV(
    estimator = rf_class,
    param_grid = param_grid,
    scoring = 'accuracy',
    n_jobs = 8,
    cv = 5,
    refit = True,
    return_train_score = True
)

# Because we set refit to True we can directly ise the object:

# Fit the object to our data
grid_rf_class.fit(X_train, y_train)

# Make predictions
grid_rf_class.predict(X_test)

/Users/gblasd/Documents/Code/EsembleML/ensembleml/lib/python3.13/site-packages/sklearn/model_selection/_validation.py:528: FitFailedWarning: 
45 fits failed out of a total of 90.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
40 fits failed with the following error:
Traceback (most recent call last):
  File "/Users/gblasd/Documents/Code/EsembleML/ensembleml/lib/python3.13/site-packages/sklearn/model_selection/_validation.py", line 866, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
    ~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/gblasd/Documents/Code/EsembleML/ensembleml/lib/python3.13/site-packages/sklearn/base.py", line 1382, in wrapper
    estimator._validate_params()
    ~~~~~~~~~~~~~~~~~~~~~~~~

array([0, 0, 0, ..., 0, 0, 0], shape=(6000,))

__Analyzing the output__

Let's analyze the GridSearchCV outputs.

Three different groups for the GridSearchCV properties:

* A results log: `cv_results_`
* The best results: `best_index_`,`best_params_` & `best_score_`
* Extra information: `scorer_`, `s_splits_` & `refit_time_`  

__Accessing object properties__

properties are accessed using the dot notation.

For example:

`grid_search_object.property`

Where `property` is the actual property you want to retrieve.


__The .cv_results_ property__

The `cv_results_` property:


In [18]:
cv_results_property = pd.DataFrame(grid_rf_class.cv_results_)
cv_results_property.shape

(18, 23)

In [20]:
cv_results_property.head()

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_max_depth,param_max_features,param_min_samples_leaf,params,split0_test_score,split1_test_score,split2_test_score,split3_test_score,split4_test_score,mean_test_score,std_test_score,rank_test_score,split0_train_score,split1_train_score,split2_train_score,split3_train_score,split4_train_score,mean_train_score,std_train_score
0,0.003443,0.001437,0.000000,0.000000,2,auto,1,"{'max_depth': 2, 'max_features': 'auto', 'min_...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,10,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,0.001644,0.000168,0.000000,0.000000,2,auto,2,"{'max_depth': 2, 'max_features': 'auto', 'min_...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,10,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,0.001778,0.001004,0.000000,0.000000,2,auto,4,"{'max_depth': 2, 'max_features': 'auto', 'min_...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,10,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,0.740650,0.027232,0.008833,0.002886,2,sqrt,1,"{'max_depth': 2, 'max_features': 'sqrt', 'min_...",0.888125,0.881875,0.931042,0.899583,0.899375,0.900000,0.016933,7,0.884896,0.881615,0.931615,0.903542,0.898333,0.900000,0.017782
4,0.738263,0.016800,0.009572,0.001799,2,sqrt,2,"{'max_depth': 2, 'max_features': 'sqrt', 'min_...",0.892292,0.888542,0.886667,0.878958,0.903333,0.889958,0.007977,8,0.889583,0.887708,0.887865,0.886198,0.902500,0.890771,0.005962


In [25]:
list(filter(lambda cadena: "time" in cadena, cv_results_property.columns))

['mean_fit_time', 'std_fit_time', 'mean_score_time', 'std_score_time']

In [26]:
cv_results_property[list(filter(lambda cadena: "time" in cadena, cv_results_property.columns))]

,mean_fit_time,std_fit_time,mean_score_time,std_score_time
0,0.003443,0.001437,0.000000,0.000000
1,0.001644,0.000168,0.000000,0.000000
2,0.001778,0.001004,0.000000,0.000000
3,0.740650,0.027232,0.008833,0.002886
4,0.738263,0.016800,0.009572,0.001799
5,0.724556,0.029464,0.010672,0.002259
6,0.002563,0.000778,0.000000,0.000000
7,0.002356,0.000533,0.000000,0.000000
8,0.001839,0.000564,0.000000,0.000000
9,1.056248,0.036220,0.010662,0.001959


In [27]:
cv_results_property[list(filter(lambda cadena: "param" in cadena, cv_results_property.columns))]

,param_max_depth,param_max_features,param_min_samples_leaf,params
0,2,auto,1,"{'max_depth': 2, 'max_features': 'auto', 'min_..."
1,2,auto,2,"{'max_depth': 2, 'max_features': 'auto', 'min_..."
2,2,auto,4,"{'max_depth': 2, 'max_features': 'auto', 'min_..."
3,2,sqrt,1,"{'max_depth': 2, 'max_features': 'sqrt', 'min_..."
4,2,sqrt,2,"{'max_depth': 2, 'max_features': 'sqrt', 'min_..."
5,2,sqrt,4,"{'max_depth': 2, 'max_features': 'sqrt', 'min_..."
6,4,auto,1,"{'max_depth': 4, 'max_features': 'auto', 'min_..."
7,4,auto,2,"{'max_depth': 4, 'max_features': 'auto', 'min_..."
8,4,auto,4,"{'max_depth': 4, 'max_features': 'auto', 'min_..."
9,4,sqrt,1,"{'max_depth': 4, 'max_features': 'sqrt', 'min_..."


In [32]:
pd.set_option("display.max_colwidth", 100)
cv_results_property.loc[:, "params"]

0     {'max_depth': 2, 'max_features': 'auto', 'min_samples_leaf': 1}
1     {'max_depth': 2, 'max_features': 'auto', 'min_samples_leaf': 2}
2     {'max_depth': 2, 'max_features': 'auto', 'min_samples_leaf': 4}
3     {'max_depth': 2, 'max_features': 'sqrt', 'min_samples_leaf': 1}
4     {'max_depth': 2, 'max_features': 'sqrt', 'min_samples_leaf': 2}
5     {'max_depth': 2, 'max_features': 'sqrt', 'min_samples_leaf': 4}
6     {'max_depth': 4, 'max_features': 'auto', 'min_samples_leaf': 1}
7     {'max_depth': 4, 'max_features': 'auto', 'min_samples_leaf': 2}
8     {'max_depth': 4, 'max_features': 'auto', 'min_samples_leaf': 4}
9     {'max_depth': 4, 'max_features': 'sqrt', 'min_samples_leaf': 1}
10    {'max_depth': 4, 'max_features': 'sqrt', 'min_samples_leaf': 2}
11    {'max_depth': 4, 'max_features': 'sqrt', 'min_samples_leaf': 4}
12    {'max_depth': 8, 'max_features': 'auto', 'min_samples_leaf': 1}
13    {'max_depth': 8, 'max_features': 'auto', 'min_samples_leaf': 2}
14    {'max_depth': 

In [34]:
cv_results_property[list(filter(lambda cadena: "test_score" in cadena, cv_results_property.columns))]

,split0_test_score,split1_test_score,split2_test_score,split3_test_score,split4_test_score,mean_test_score,std_test_score,rank_test_score
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,10
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,10
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,10
3,0.888125,0.881875,0.931042,0.899583,0.899375,0.900000,0.016933,7
4,0.892292,0.888542,0.886667,0.878958,0.903333,0.889958,0.007977,8
5,0.877708,0.883750,0.867917,0.901458,0.859375,0.878042,0.014367,9
6,NaN,NaN,NaN,NaN,NaN,NaN,NaN,10
7,NaN,NaN,NaN,NaN,NaN,NaN,NaN,10
8,NaN,NaN,NaN,NaN,NaN,NaN,NaN,10
9,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,0.000000,1


In [40]:
# Read the cv_results property into a dataframe & print it out
cv_results_df = pd.DataFrame(grid_rf_class.cv_results_)
print(cv_results_df)

    mean_fit_time  std_fit_time  ...  mean_train_score  std_train_score
0        0.003443      0.001437  ...               NaN              NaN
1        0.001644      0.000168  ...               NaN              NaN
2        0.001778      0.001004  ...               NaN              NaN
3        0.740650      0.027232  ...          0.900000         0.017782
4        0.738263      0.016800  ...          0.890771         0.005962
5        0.724556      0.029464  ...          0.878510         0.015441
6        0.002563      0.000778  ...               NaN              NaN
7        0.002356      0.000533  ...               NaN              NaN
8        0.001839      0.000564  ...               NaN              NaN
9        1.056248      0.036220  ...          1.000000         0.000000
10       1.031247      0.033595  ...          1.000000         0.000000
11       1.071141      0.054224  ...          1.000000         0.000000
12       0.001440      0.000675  ...               NaN          

In [41]:
# Extract and print the column with a dictionary of hyperparameters used
column = cv_results_df.loc[:, ["params"]]
print(column)

                                                             params
0   {'max_depth': 2, 'max_features': 'auto', 'min_samples_leaf': 1}
1   {'max_depth': 2, 'max_features': 'auto', 'min_samples_leaf': 2}
2   {'max_depth': 2, 'max_features': 'auto', 'min_samples_leaf': 4}
3   {'max_depth': 2, 'max_features': 'sqrt', 'min_samples_leaf': 1}
4   {'max_depth': 2, 'max_features': 'sqrt', 'min_samples_leaf': 2}
5   {'max_depth': 2, 'max_features': 'sqrt', 'min_samples_leaf': 4}
6   {'max_depth': 4, 'max_features': 'auto', 'min_samples_leaf': 1}
7   {'max_depth': 4, 'max_features': 'auto', 'min_samples_leaf': 2}
8   {'max_depth': 4, 'max_features': 'auto', 'min_samples_leaf': 4}
9   {'max_depth': 4, 'max_features': 'sqrt', 'min_samples_leaf': 1}
10  {'max_depth': 4, 'max_features': 'sqrt', 'min_samples_leaf': 2}
11  {'max_depth': 4, 'max_features': 'sqrt', 'min_samples_leaf': 4}
12  {'max_depth': 8, 'max_features': 'auto', 'min_samples_leaf': 1}
13  {'max_depth': 8, 'max_features': 'auto', 'mi

In [42]:
# Extract and print the row that had the best mean test score
best_row = cv_results_df[cv_results_df["rank_test_score"] == 1 ]
print(best_row)

    mean_fit_time  std_fit_time  ...  mean_train_score  std_train_score
9        1.056248      0.036220  ...               1.0              0.0
10       1.031247      0.033595  ...               1.0              0.0
15       1.462860      0.105206  ...               1.0              0.0
16       1.457899      0.044722  ...               1.0              0.0
17       1.398291      0.052115  ...               1.0              0.0

[5 rows x 23 columns]


__The best grid square__

Information on the best grid square is neatly summarized in the following three properties:

* `best_params_`, the dictionary of parameters that gave the best score.
* `best_score_`, the actual best score.
* `best_index_`, the row in our `cv_results_.rank_test_score` thet was the best.

__The best_estimator\_ property__

The `best_estimator_` property is an estimator build using the best parameters from the grid search.

For us this is a Random Forest estimator:

`type(grid_rf_class.best_estimator_)`

`sklearn.ensemble.forest.RandomForestClassifier`

We could also directly use this object as an estimator if we want!

`print(grid_rf_class.best_estimator_)`

In [35]:
grid_rf_class.best_estimator_

RandomForestClassifier(criterion='entropy', max_depth=4)

In [37]:
grid_rf_class.best_params_

{'max_depth': 4, 'max_features': 'sqrt', 'min_samples_leaf': 1}

In [45]:
# Get the n_estimators parameter from the best-performing square and print
best_n_estimators = grid_rf_class.best_params_["max_features"]
print(best_n_estimators)

sqrt


__Extra information__

Some extra information is available in the following properties:

* `scorer_` What scorer function was used on the held out data. (We set it to AUC)

* `s_splits_` How many cross-validation splits.


In [38]:
grid_rf_class.scorer_

make_scorer(accuracy_score, response_method='predict')

In [ ]:
grid_rf_class.best_score_ # ROC_AUC score from the best-performing square, if this metric is defined in the train

np.float64(1.0)

In [46]:
# See what type of object the best_estimator_ property is
print(type(grid_rf_class.best_estimator_))

<class 'sklearn.ensemble._forest.RandomForestClassifier'>


In [47]:
# Create an array of predictions directly using the best_estimator_ property
predictions = grid_rf_class.best_estimator_.predict(X_test)

In [48]:
# Take a look to confirm it worked, this should be an array of 1's and 0's
print(predictions[0:5])

[0 0 0 0 1]


In [49]:
from sklearn.metrics import confusion_matrix, roc_auc_score

In [50]:
# Now create a confusion matrix 
print("Confusion Matrix \n", confusion_matrix(y_test, predictions))

Confusion Matrix 
 [[4687    0]
 [   1 1312]]


In [51]:
# Get the ROC-AUC score
predictions_proba = grid_rf_class.best_estimator_.predict_proba(X_test)[:,1]
print("ROC-AUC Score \n", roc_auc_score(y_test, predictions_proba))

ROC-AUC Score 
 1.0
